# Lanes Maps

The following code aggregates the processing of the previous notebook (`8_Cycling_Potential_Lanes_Processing`) and display it visually in form of maps.

The maps show the roads (streets, avenues) of São Paulo, filtering by modal (vehicle) and cyclability.

In [1]:
%load_ext autoreload
%autoreload 2
import saopaulo.load_trips as sptr
import saopaulo.maps_aux as aux
import saopaulo.cycling_infrastructure as cinfra
import saopaulo.choropleth as spchoro
import bikescience.map_widgets as mpw
from math import log

import datetime

from shapely.geometry import LineString, Point
#import saopaulo.choropleth_folium as spchoro

import saopaulo.sp_grid as gr

from bikescience.intersect_ways import geometry_intersection_length
import datetime
import geopandas as gpd
import json
import pandas as pd
import numpy as np
from ipywidgets import interact_manual, widgets, fixed
from IPython.core.display import display, HTML, clear_output
import folium

import warnings
import requests
import os
import fnmatch as fnm
warnings.simplefilter('ignore')

gr.SP_LAT = -23.63
gr.SP_LON = -46.55


/home/pedro/.local/lib/python3.8/site-packages/geopandas/_compat.py:106: UserWarning: The Shapely GEOS version (3.8.0-CAPI-1.13.1 ) is incompatible with the GEOS version PyGEOS was compiled with (3.9.0-CAPI-1.16.2). Conversions between both will be slow.
  warnings.warn(


In [2]:
# Load shp of bicycle lanes
protected_lanes = cinfra.load_protected_lanes('../data/sao-paulo/infraestrutura-cicloviaria/cet/Ciclovias.shp')
bike_lanes = cinfra.load_bike_lanes('../data/sao-paulo/infraestrutura-cicloviaria/cet/Ciclovias.shp')
infra = pd.concat([protected_lanes, bike_lanes])

In [3]:
districts_names = [
    'Bela Vista', 'Bom Retiro', 'Cambuci', 'Consolação', 'Liberdade', 
    'República', 'Santa Cecília', 'Sé','Butantã', 'Morumbi', 'Raposo Tavares', 
    'Rio Pequeno', 'Vila Sônia', 'Barra Funda', 'Jaguara', 'Jaguaré', 'Lapa', 
    'Perdizes', 'Vila Leopoldina', 'Alto de Pinheiros', 'Itaim Bibi', 
    'Jardim Paulista', 'Pinheiros','Aricanduva', 'Carrão', 'Vila Formosa',
    'Água Rasa', 'Belém', 'Brás', 'Mooca', 'Pari', 'Tatuapé', 'Artur Alvim',
    'Cangaíba', 'Penha', 'Vila Matilde', 'São Lucas', 'Sapopemba',
    'Vila Prudente','Cidade Tiradentes', 'Ermelino Matarazzo', 'Ponte Rasa',
    'Guaianases', 'Lajeado', 'Itaim Paulista', 'Vila Curuçá', 'Cidade Líder',
    'Itaquera', 'José Bonifácio', 'Parque do Carmo', 'Iguatemi', 'São Mateus',
    'São Rafael', 'Jardim Helena', 'São Miguel', 'Vila Jacuí','Cachoeirinha', 
    'Casa Verde', 'Limão', 'Brasilândia', 'Freguesia do Ó', 'Anhanguera', 
    'Perus', 'Jaraguá', 'Pirituba', 'São Domingos','Jaçanã', 'Tremembé', 
    'Mandaqui', 'Santana', 'Tucuruvi', 'Vila Guilherme', 'Vila Maria', 
    'Vila Medeiros','Cursino', 'Ipiranga', 'Sacomã', 'Jabaquara', 'Moema', 
    'Saúde', 'Vila Mariana', 'Campo Limpo', 'Capão Redondo', 'Vila Andrade', 
    'Cidade Dutra', 'Grajaú', 'Socorro', 'Cidade Ademar', 'Pedreira', 
    'Jardim Ângela', 'Jardim São Luís', 'Marsilac', 'Parelheiros',
    'Campo Belo', 'Campo Grande', 'Santo Amaro']

In [19]:
sao_paulo = gpd.read_file('../data/sao-paulo/SP_Municipios_2021/SP_Municipios_2021.shp')
sao_paulo = sao_paulo.loc[sao_paulo['NM_MUN'] == 'São Paulo']

zone_shp = gpd.read_file('../data/sao-paulo/od/shapes/Distritos_2017_region.shp')
zone_shp.crs = {'init': 'epsg:31983'}  
zone_shp.to_crs(epsg='4326', inplace=True)
zone_shp['NumeroMuni'] = [36 if z['NomeDistri'] in districts_names else 0 for _, z in zone_shp.iterrows()]

In [7]:
# the previous notebook calculated the number of trips in each lane
# but the processing was made per district - now we need to join all the districts
def concatenate_lanes_districts():
    lanes = []
    lanes_faltando = []
    for d in distritos:
        try:
            lanes.append(gpd.read_file('bases/Potential_V2/potential_lanes_' + d + '.shp'))
        except Exception as e:
            lanes_faltando.append(e)
    print(len(lanes))
    print(len(lanes_faltando))
    print(lanes_faltando)
    lanes = pd.concat(lanes)
    print(len(lanes))
    lanes.to_file('bases/lanes_pocessed/lanes_processed.shp')

# process the lanes vs the current cycling infrastructure to discover the regions not yet served by bike lanes    
def calculate_lanes_far_from_infra():
    longe_ciclovia = []
    progress = 0
    i = 0
    print(len(lanes))
    for _, lane in lanes.iterrows():
        if i / len(lanes) >= progress:
            print (round(progress * 100), '%  - ', datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
            progress += 0.05
        i += 1
        #print (datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"), ' - ', i)
        longe = True
        for _, j in infra.iterrows():
            distance = j['geometry'].distance(lane['geometry'])
            if distance <= 0.002:
                #print(lane['lg_nome'])
                #print(j['programa'])
                #print(distance)
                #print('--------------------')
                longe = False
                break
        longe_ciclovia.append(longe)

        free_lanes = lanes.iloc[longe_ciclovia]
        free_lanes.to_file('bases/lanes_without_infra/lanes_without_infra.shp')

In [6]:
try:
    lanes = gpd.read_file('bases/lanes_processed/lanes_processed.shp')
except:
    print('File lanes_processed not found')
    print('processing it - may take a few minutes')
    concatenate_lanes_districts()
    lanes = gpd.read_file('bases/lanes_processed/lanes_processed.shp')
    
print(len(lanes))

226775


In [9]:
try:
    free_lanes = gpd.read_file('bases/lanes_without_infra/lanes_without_infra.shp')
except:
    print('File lanes_without_infra not found')
    print('processing it - may take a few hours')
    calculate_lanes_far_from_infra()
    free_lanes = gpd.read_file('bases/lanes_without_infra/lanes_without_infra.shp')
    
print(len(free_lanes))  

162943


In [10]:
# some columns lanes are corrupted int the file (dont know why)

modals = ['pedestrian', 'subway', 'bus_sp', 'taxi_app', 'car', 'bus_private',
       'train', 'taxi', 'bus_school', 'other', 'motorcycle', 'bus_inter',
       'bus_otherc', 'monorail']

columns_mapping = {'potential_': 'potential_pedestrian_0'}

i = 0
for m in modals:
    for q in range(4):
        column = 'potentia'
        if i < 10: column += '_'
        column += str(i)
        if i == 0: column = 'potential_'
        columns_mapping[column] = 'potential_' + m + '_' + str(q)
        i += 1   
        
lanes.rename(columns_mapping, axis='columns', inplace = True)
free_lanes.rename(columns_mapping, axis='columns', inplace = True)

In [11]:
colors = ['#ffffb2','#fed976','#feb24c','#fd8d3c','#f03b20', '#bd0026'] #YlOrRd
colors = ['#a1d99b','#74c476','#41ab5d','#238b45','#006d2c','#00441b'] #green
colors = ['#ED553B','#F6D55C','#3CAEA3','#20639B'] #multi color
colors = ['#E1F5FE','#B3E5FC','#81D4FA','#4FC3F7','#29B6F6','#03A9F4','#039BE5','#0288D1','#0277BD','#01579B'] #blue
colors = ['#FEF001','#FFCE03','#FD9A01','#FD6104','#FF2C05','#F00505'] #Yellow to Red

n = len(colors)
intervals     = [i/n for i in range(1, n + 1)]                # [0.20, 0.40, 0.60, 0.80, 1]
intervals_sq  = [(i/n)**2 for i in range(1, n + 1)]           # [0.04, 0.16, 0.36, 0.64, 1] detalha valores baixos
intervals_log = [log(i + 1, n + 1) for i in range(1, n + 1)]  # [0.38, 0.61, 0.77, 0.89, 1] detalha valores altos

intervals = intervals_sq

round_factor = 100

def get_color (potential, min_trip, max_trip):
    for i in range(n):
        if potential <= min_trip + intervals[i] * (max_trip - min_trip): return colors[i];

def build_subtitle (min_trip, max_trip):
    subtitle = []
    for i in range(n):
        subtitle.append([colors[i], round((min_trip + intervals[i] * (max_trip - min_trip)) / round_factor) * round_factor])
    subtitle.reverse()
    return subtitle

In [33]:
fmap = gr.map_around_sp(the_grid=None,zoom=10, plot_grid=False)
#style_infra = lambda style:{'color':'blue', 'weight': 1, 'opacity': 0.5 }
#folium.GeoJson(infra, show = False, control = True,
#    style_function = style_infra,
#    name = 'bike lanes').add_to(fmap)
#max_trip = max(lanes[column])

style_sao_paulo = lambda style:{'color':'black', 'weight': 0.8, 'fill': False }

folium.GeoJson(sao_paulo, style_sao_paulo).add_to(fmap)
display(fmap)

In [2]:
def add_infra_layer(fmap):
    style_infra = lambda style:{'color':'blue', 'weight': 2 }
    folium.GeoJson(infra, show = False, control = True,
        style_function = style_infra,
        name = 'bike lanes').add_to(fmap)
    folium.LayerControl().add_to(fmap)

def plot_potential_lanes(lanes, title = "", display_map = True, show_greater_than = .05, plot_infra = True, 
                         column = None):
        
    fmap = gr.map_around_sp(the_grid=None,zoom=10, plot_grid=False)
            
    max_trip = max(lanes[column])
    lanes_aux = lanes.loc[lanes[column] > show_greater_than*max_trip ]
    min_trip = min(lanes_aux[column])
    
    #spchoro.plot_zones_border(fmap, zone_shp, '#333333', plot_rmsp = False, weight = 0.7)
    
    style_sao_paulo = lambda style:{'color':'black', 'weight': 1.5, 'fill': False }
    folium.GeoJson(sao_paulo, style_sao_paulo, control = False).add_to(fmap)
    
    style_lanes = lambda x : {'color':get_color(x['properties'][column], min_trip, max_trip), 
                              'weight': 1 + (5 * x['properties'][column] / max_trip), 'opacity': 0.95}
    folium.GeoJson(lanes_aux, style_lanes, control=False).add_to(fmap)
    
    if (plot_infra):
        add_infra_layer(fmap)
    
    fmap.get_root().html.add_child(folium.Element(mpw.build_title('Cyclable Trips')))    
    fmap.get_root().html.add_child(folium.Element(mpw.build_legend_lines('Trips', build_subtitle(min_trip, max_trip))))
    fmap.get_root().html.add_child(folium.Element(mpw.drag_function('Trips')))
    
    fmap.save('maps/per_modal_tier/' + title + '.html')
    if (display_map):
        display(fmap)

In [3]:
def build_title_modals_tiers(pedestrian, car, motorcycle, bus, subway, train, tiers, far_from_infra):
    title = ''
    for t in tiers:
        title += str(t) + '_'
    if pedestrian:
        title += 'ped_'
    if car:
        title += 'car_'
    if motorcycle:
        title += 'mot_'
    if bus:
        title += 'bus_'
    if subway:
        title += 'sub_'
    if train:
        title += 'tra_'
        
    if far_from_infra:
        title += 'far_from_infra_'
    return title[:-1]    
        
def generate_potential_df(
    lanes_aux, pedestrian, car, motorcycle, bus, subway, train,
    tiers):
    potential = pd.Series([0]*len(lanes_aux))
    for t in tiers:
        if pedestrian:
            potential += pd.to_numeric(lanes_aux['potential_pedestrian_' + str(t)].fillna(0))
        if car:
            potential += pd.to_numeric(lanes_aux['potential_car_' + str(t)].fillna(0))
            potential += pd.to_numeric(lanes_aux['potential_taxi_' + str(t)].fillna(0))
            potential += pd.to_numeric(lanes_aux['potential_taxi_app_' + str(t)].fillna(0))
        if motorcycle:
            potential += pd.to_numeric(lanes_aux['potential_motorcycle_' + str(t)].fillna(0))
        if bus:
            potential += pd.to_numeric(lanes_aux['potential_bus_sp_' + str(t)].fillna(0))
            potential += pd.to_numeric(lanes_aux['potential_bus_inter_' + str(t)].fillna(0))
            potential += pd.to_numeric(lanes_aux['potential_bus_otherc_' + str(t)].fillna(0))
            potential += pd.to_numeric(lanes_aux['potential_bus_school_' + str(t)].fillna(0))
            potential += pd.to_numeric(lanes_aux['potential_bus_private_' + str(t)].fillna(0))
        if subway:
            potential += pd.to_numeric(lanes_aux['potential_subway_' + str(t)].fillna(0))
            potential += pd.to_numeric(lanes_aux['potential_monorail_' + str(t)].fillna(0))
        if train:
            potential += pd.to_numeric(lanes_aux['potential_train_' + str(t)].fillna(0))
    return potential
    
def generate_map(
    pedestrian, car, motorcycle, bus, subway, train,
    tiers, display_map=False, far_from_infra = False):
    
    lanes_aux = free_lanes if far_from_infra else lanes
            
    lanes_aux['potential_aggregate'] = generate_potential_df(lanes_aux, pedestrian, car, motorcycle, bus, subway, train, tiers)
    
    title = build_title_modals_tiers(pedestrian, car, motorcycle, bus, subway, train, tiers, far_from_infra)
    
    plot_potential_lanes(lanes_aux, title = title, display_map = display_map, show_greater_than = .1, 
                         plot_infra = True, 
                         column = 'potential_aggregate')
            

In [42]:
#Example of map
generate_map(
    pedestrian = False,
    car = False, 
    motorcycle = True, 
    bus = False, 
    subway = False, 
    train = False,
    tiers = [3], 
    display_map = True,
    far_from_infra = False)

In [43]:
# generate all 2^10 possible maps - turns out some maps are too heavy and I dont have enough storage
def generate_all_maps():
    i = 0
    total = (2**4 - 1)*(2**6 - 1)

    for tiers in range(1, 2**4):

        tier_bit_array = '{0:b}'.format(tiers).zfill(4)
        tiers = []
        for t in range(4):
            if tier_bit_array[t] == '1':
                tiers.append(t)

        for modals in range(1, 2**6):
            modal_bit_array = '{0:b}'.format(modals).zfill(6)
            modal_bits = []
            for bit in modal_bit_array:
                modal_bits.append(bit == '1')

            print(datetime.datetime.now(), '  ', str(i).zfill(3) + ' / ' + str(total), '   -    ', tier_bit_array, modal_bit_array)
            i += 1     

            generate_map(
                modal_bits[0], modal_bits[1], modal_bits[2], 
                modal_bits[3], modal_bits[4], modal_bits[5],
                tiers)


In [4]:
def generate_map_widgets_pamareters(
    pedestrian, car, motorcycle, bus, subway, train,
    tier0, tier1, tier2, tier3, far_from_infra):
    tiers = []
    if tier0: 
        tiers.append(0)
    if tier1: 
        tiers.append(1)
    if tier2: 
        tiers.append(2)
    if tier3: 
        tiers.append(3)
        
    generate_map(pedestrian, car, motorcycle, bus, subway,
                 train, tiers, True, far_from_infra)

def generate_map_widgets():
    
    width = '200px'
    layout = widgets.Layout(width = '500px')
    
    im = interact_manual(
        generate_map_widgets_pamareters,
        pedestrian = widgets.Checkbox(description = 'Pedestrian'),
        car = widgets.Checkbox(description = 'Car'),
        motorcycle = widgets.Checkbox(description = 'Motorcycle'),
        bus = widgets.Checkbox(description = 'Bus'),
        subway = widgets.Checkbox(description = 'Subway'),
        train = widgets.Checkbox(description = 'Train'),
        tier0 = widgets.Checkbox(description = 'Very low ciclability'),
        tier1 = widgets.Checkbox(description = 'Low ciclability'),
        tier2 = widgets.Checkbox(description = 'High ciclability'),
        tier3 = widgets.Checkbox(description = 'Very high ciclability'),
        far_from_infra = widgets.Checkbox(description = 'Far from bicycle lanes')
    )
    im.widget.children[11].description = 'Generate Map'
    
generate_map_widgets()

interactive(children=(Checkbox(value=False, description='Pedestrian'), Checkbox(value=False, description='Car'…

# Bicycle trips

In [49]:
lanes_bike = []
for d in districts_names:
    try:
        lanes_bike.append(gpd.read_file('bases/Bicycle_Trips_Per_Lane/bike_trips_per_lanes_' + d + '.shp'))
    except Exception as e:
        print(d, e)
        
lanes_bike = pd.concat(lanes_bike)
print(len(lanes_bike))

226775


In [54]:
plot_potential_lanes(lanes_bike, "Bicycle trips per lanes", True, show_greater_than = .1, column="bicycle_tr")